In [1]:
import pandas as pd
import numpy as np

df=pd.read_csv("training.1600000.processed.noemoticon.csv",encoding='ISO-8859-1',header=None)
df.head()

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [2]:
df.columns =['sentiment','id','date','ifquery','username','tweetcontent']
df.head()

,sentiment,id,date,ifquery,username,tweetcontent
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [3]:
df['tweetcontent']=df['tweetcontent'].str.lower()
df['tweetcontent']=df['tweetcontent'].str.replace(r'#','',regex=True)
df['tweetcontent']=df['tweetcontent'].str.replace(r'@\w+','',regex=True)
df['tweetcontent']=df['tweetcontent'].str.replace(r'https?://\S+','', regex=True)
df['tweetcontent']=df['tweetcontent'].str.replace(r'[^\w\s]','', regex=True)
df['tweetcontent']=df['tweetcontent'].str.replace(r'\s+',' ', regex=True)
df['tweetcontent']=df['tweetcontent'].str.strip() 
df.head()


,sentiment,id,date,ifquery,username,tweetcontent
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,awww thats a bummer you shoulda got david carr...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he cant update his facebook by t...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,i dived many times for the ball managed to sav...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,no its not behaving at all im mad why am i her...


In [4]:
map_strlabels={0:'Negative',2:'Neutral',4:'Positive'}
map_numlabels={0:-1,2:0,4:1}
df['sentiment']=df['sentiment'].map(map_numlabels)
df.head()

,sentiment,id,date,ifquery,username,tweetcontent
0,-1,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,awww thats a bummer you shoulda got david carr...
1,-1,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he cant update his facebook by t...
2,-1,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,i dived many times for the ball managed to sav...
3,-1,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,-1,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,no its not behaving at all im mad why am i her...


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=5000)
tweets_after_tfidf=vectorizer.fit_transform(df['tweetcontent'])

In [6]:
print(tweets_after_tfidf.shape)

(1600000, 5000)


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

x=tweets_after_tfidf
y=df['sentiment']
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=42)


Model selection:
- high-dimensional data(SVM possible)
- data trained with tf-idf so, matrix has sparse linear values.
- sentiment only has 3 values so Logistic Regression is possible

-Implementing cross validation to compare models


model2=SVC(kernel='linear',C=0.1,gamma='scale',random_state=42)
score_svm=cross_val_score(model2,x,y,cv=5,scoring='accuracy')
print("cross validation accuracy linear svm:",score_svm.mean())

- This was too slow on this large dataset

In [9]:
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC,LinearSVC
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier

model1=LogisticRegression(random_state=42)
score_log=cross_val_score(model1,x,y,cv=5,scoring='accuracy')
print("cross validation accuracy LogisticRegression:",score_log.mean())

model2=LinearSVC(C=0.1,random_state=42)
score_svm=cross_val_score(model2,x,y,cv=5,scoring='accuracy')
print("cross validation accuracy linear svm:",score_svm.mean())

model3=MultinomialNB()
score_naivebayes=cross_val_score(model3,x,y,cv=5,scoring='accuracy')
print("cross validation accuracy naive bayes:",score_naivebayes.mean())




cross validation accuracy LogisticRegression: 0.7863174999999999
cross validation accuracy linear svm: 0.7859149999999999
cross validation accuracy naive bayes: 0.7637962500000001


model4=XGBClassifier(random_state=42)
score_xgb=cross_val_score(model4,x,y,cv=5,scoring='accuracy')
print("cross validation accuracy naive bayes:",score_xgb.mean())

- Here, y should be either 0 or 1. Here, y can be -1,0 or 1. 
- Hence converting -1 to 0 here for model training. 

In [14]:
model4=XGBClassifier(random_state=42,use_label_encoder=False,n_estimators=50,max_depth=3)
import warnings
warnings.filterwarnings(action='ignore')
y_for_xgb=y.copy()
y_for_xgb=y_for_xgb.replace(-1, 0)
score_xgb=cross_val_score(model4,x,y_for_xgb,cv=5,scoring='accuracy')
print("cross validation accuracy xgb:",score_xgb.mean())

cross validation accuracy naive bayes: 0.714104375


Logistic Regression was most accurate and fast enough.

Model1 already trained on data.

In [19]:
print("Model selected is logistic regression")
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
model1.fit(xtrain,ytrain)
ypredict_model1=model1.predict(xtest)
print("Accuracy:",accuracy_score(ytest,ypredict_model1))
cm=confusion_matrix(ytest,ypredict_model1)
print(cm)
print(classification_report(ytest,ypredict_model1))

Model selected is logistic regression
Accuracy: 0.790478125
[[123881  35613]
 [ 31434 129072]]
              precision    recall  f1-score   support

           0       0.80      0.78      0.79    159494
           1       0.78      0.80      0.79    160506

    accuracy                           0.79    320000
   macro avg       0.79      0.79      0.79    320000
weighted avg       0.79      0.79      0.79    320000

